Trying to extract the most prominent k-mers from the reactive, bystander, self tolerant, and pre-immune datasets to compare what is the most prevalent marker of a tcr from that item

Imports

In [1]:
import polars as pl
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

Path setting

In [2]:
data_dir_kb = Path("../Data/20250910 Comparison 2/Kb")
output_dir = Path("Comparison2_v2")
output_dir.mkdir(exist_ok=True)

Reading kb files

In [3]:
kb_reactive = pl.concat([pl.read_csv(data_dir_kb / f"20250910 B10BR PD1hi{x} LL TCR Repertoire.csv") for x in 'ABCDE'])

kb_selfTolerant = pl.concat([pl.read_csv(data_dir_kb / "20250910 1783 Naive LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 1783 Naive SLO TCR Repertoire.csv")])

kb_bystander = pl.read_csv(data_dir_kb / "20250910 B10BR PD1negD LL TCR Repertoire.csv")

kb_preImmune = pl.concat([pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR pre-immmune TCR Repertoire.csv")])

In [4]:
TRAVS = ['TRAV13', 'TRAV14', 'TRAV16']

def trav_label(df):
    '''Adds a column that details where a cell has either TRAV13/14/16
    Args:
        df - pl.DataFrame
    Returns:
        df - pl.DataFrame
    '''

    return df.with_columns(pl.col('TRAV').str.extract(r'^(TRAV\d+)').alias('TRAV_group'))

In [5]:
kb_reactive = trav_label(kb_reactive)
kb_selfTolerant = trav_label(kb_selfTolerant)
kb_preImmune = trav_label(kb_preImmune)

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kb trav

In [6]:
kb_reactive_trav_counts = kb_reactive['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (87,2)
kb_selfTolerant_trav_counts = kb_selfTolerant['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (143,2)
kb_preImmune_trav_counts = kb_preImmune['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (90,2)

kb_trav = (kb_reactive_trav_counts
            .join(kb_selfTolerant_trav_counts, on='TRAV', how='full', coalesce=True)
            .join(kb_preImmune_trav_counts, on='TRAV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kb_trav_freq = kb_trav.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kb_trav.write_csv(output_dir / 'kb_trav_counts')
kb_trav_freq.write_csv(output_dir / 'kb_trav_freqs')

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kb trbv

In [7]:
kb_reactive_trbv_counts = kb_reactive['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (23,2)
kb_selfTolerant_trbv_counts = kb_selfTolerant['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (25,2)
kb_preImmune_trbv_counts = kb_preImmune['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (25,2)

kb_trbv = (kb_reactive_trbv_counts
            .join(kb_selfTolerant_trbv_counts, on='TRBV', how='full', coalesce=True)
            .join(kb_preImmune_trbv_counts, on='TRBV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kb_trbv_freq = kb_trbv.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kb_trbv.write_csv(output_dir / 'kb_trbv_counts')
kb_trbv_freq.write_csv(output_dir / 'kb_trbv_freqs')

Return all k-mers and their counts

In [8]:
def extract_kmers(df, col_name, out_alias, k):
    """Return all k-mers from an amino-acid sequence.
    Args:
        df - pl.DataFrame
        col_name - str,
        out_alias - str,
        k - int
    Returns:
        pl.DataFrame
    """
    return(
        df.select(
            pl.concat_list([
                pl.col(col_name).str.slice(i, k)
                for i in range(df[col_name].str.len_chars().max() - k + 1)
            ]).alias(f'{k}-mer')
        )
        .explode(f'{k}-mer', empty_as_null=True)
        .filter(pl.col(f'{k}-mer').str.len_chars() == k)
        .group_by(f'{k}-mer')
        .len(name=out_alias)
    )

one dataframe for everything x

In [9]:
def counts(df_list, col_name, aliases, k):
    """Merge all the counts into one table
    Args:
        df_list - list
        col_name - str
        aliases - list
        k - int
    Returns:
        pl.DataFrame"""
    basis = extract_kmers(df_list[0], col_name, aliases[0], k)
    for df, alias in zip(df_list[1:], aliases[1:]):
        counts = extract_kmers(df, col_name, alias, k)
        basis = basis.join(counts, on=f'{k}-mer', how='full', coalesce=True)
    return basis.fill_null(0)

markers

In [10]:
alpha = 'CDR3a_aa'
beta = 'CDR3b_aa'

defintion to subsample set by strata

In [11]:
def stratified_subsampling(reactive_counts, preImmune_df, gene):
    '''Takes a reactive and preimmune dataframe for a single category and
    subsamples the preimmune dataframe down to the size and strata of the
    reactive dataset
    
    Args:
        reactive_counts - pl.DataFrame
        preImmune_df - pl.DataFrame
        gene - str
    Returns:
        stratified_pI - pl.DataFrame
    '''

    subsamples = []

    for row in reactive_counts.iter_rows(named=True):

        pI_pool = preImmune_df.filter(pl.col(gene) == row[gene])
        total = pI_pool.height
        sample_n = min(row['counts_r'], total)

        if sample_n > 0:
            taken = pI_pool.sample(n=sample_n, with_replacement=False, seed=1)
            subsamples.append(taken)

    stratified_pI = pl.concat(subsamples)

    return stratified_pI

In [12]:
strat_pI = {}
grouped_r = {}
grouped_sT = {}
grouped_pI = {}

for group in TRAVS:
    grouped_r[group] = kb_reactive.filter(pl.col('TRAV_group') == group)
    grouped_sT[group] = kb_selfTolerant.filter(pl.col('TRAV_group') == group)
    grouped_pI[group] = kb_preImmune.filter(pl.col('TRAV_group') == group)

    kb_reactive_trav_counts = grouped_r[group]['TRAV'].value_counts().rename({'count': 'counts_r'})

    strat_pI[group] = stratified_subsampling(kb_reactive_trav_counts, grouped_pI[group], 'TRAV')



Counting all 3-mers across the different datasets

In [13]:
kb_cdr3a_counts = {}
kb_cdr3b_counts = {}

for group in TRAVS:
    kb_cdr3a_counts[group] = counts(
        [grouped_r[group], grouped_sT[group], strat_pI[group]], alpha,
        [f'{group}_kb_reactive_cdr3a', f'{group}_kb_selfTolerant_cdr3a', f'{group}_kb_preImmune_cdr3a'], 3
    )

    kb_cdr3b_counts[group] = counts(
        [grouped_r[group], grouped_sT[group], strat_pI[group]], beta,
        [f'{group}_kb_reactive_cdr3b', f'{group}_kb_selfTolerant_cdr3b', f'{group}_kb_preImmune_cdr3b'], 3
    )

    kb_cdr3a_counts[group].write_csv(output_dir / f'{group}_kb_cdr3a_3mer_counts.csv')
    kb_cdr3b_counts[group].write_csv(output_dir / f'{group}_kb_cdr3b_3mer_counts.csv')

Z scores of the 3-mers, relative to the row average - i.e. does this 3-mer appear more or less often than the average amount

Fold change of the 3-mer relative to the pre-immune dataset, expecting the key 3-mers to be up-regulated in the reactive and down-regulated in the selfTolerant

In [14]:
def freq_and_z(df, k):
    """Computes the frequencies of all the k-mers and then calculates the relative z-scores to compare between
        reactive, self tolerant, and pre immune datasets
    Args:
        df -> pl.DataFrame
    Returns:
        tuple[pl.DataFrame, pl.DataFrame]"""
    cols = [c for c in df.columns if c != f'{k}-mer']

    freq_df = df.with_columns([pl.col(c) / pl.col(c).sum() for c in cols])
    row_mean = pl.mean_horizontal(cols)
    variance_sum = pl.sum_horizontal([(pl.col(c) - row_mean) ** 2 for c in cols])
    row_std = (variance_sum / (len(cols) - 1)).sqrt()

    zscore_df = freq_df.with_columns([
        ((pl.col(c) - row_mean) / row_std).alias(c) for c in cols
    ])

    return freq_df, zscore_df

In [15]:
kb_cdr3a_freq = {}
kb_cdr3b_freq = {}
kb_cdr3a_z = {}
kb_cdr3b_z = {}

for group in TRAVS:
    kb_cdr3a_freq[group], kb_cdr3a_z[group] = freq_and_z(kb_cdr3a_counts[group], 3)
    kb_cdr3b_freq[group], kb_cdr3b_z[group] = freq_and_z(kb_cdr3b_counts[group], 3)

Making a heatmap to visualise the z score - picked the 3-mers with the highest variance

In [16]:
datasets = []
for group in TRAVS:
    datasets.append((f'{group}_cdr3a', kb_cdr3a_freq[group], kb_cdr3a_z[group]))
    datasets.append((f'{group}_cdr3b', kb_cdr3b_freq[group], kb_cdr3b_z[group]))

for name, freq_df, z_df in datasets:
    cols = [c for c in freq_df.columns if c != '3-mer']

    most_variable = (freq_df.select([
        pl.col('3-mer'),
        (pl.sum_horizontal([pl.col(c) - pl.mean_horizontal(cols) ** 2 for c in cols]) / (len(cols) - 1)).alias('var')
        ])
        .sort('var', descending=True)
        .head(500)
        .join(z_df, on='3-mer', how='inner')
        .drop('var')
        )
    
    labels = most_variable['3-mer'].to_numpy()
    numbers = most_variable.select(cols).to_numpy()
    
    sns_df = pd.DataFrame(numbers, columns=cols, index=labels)

    sns.clustermap(sns_df, cmap='vlag', center=0, figsize=(10,70), yticklabels=True)
    plt.savefig(output_dir / f'{name}_zscore_heatmap.png')
    plt.close()